# Live / Paper Trading Performance

Queries `trading_system.db` directly to analyse:
- Cumulative P&L and open positions
- Signal quality: does higher confidence actually win more?
- Source attribution: which data source drives wins (sentiment / politician / analyst)
- Ghost trade regret: what would have been made if every signal fired
- Pending approval history
- Adanos API budget burn rate

**Running locally against a GCP database:**
```bash
# Copy the DB from GCP to your laptop first
gcloud compute scp quant-lab:~/quant-lab/trading_system.db ./trading_system.db --zone=us-central1-a
```
Then set `DB_PATH` below to `"../trading_system.db"` (relative to this notebook).

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Point this at your local copy of the database
DB_PATH = "../trading_system.db"

if not Path(DB_PATH).exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}.\n"
        "Copy it from GCP with:\n"
        "  gcloud compute scp quant-lab:~/quant-lab/trading_system.db "
        "./trading_system.db --zone=us-central1-a"
    )

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Connected to: {Path(DB_PATH).resolve()}")

## 1. Database snapshot

In [ ]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print("Tables:", tables["name"].tolist())

for table in tables["name"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"][0]
    print(f"  {table}: {count} rows")

## 2. All signals

In [ ]:
signals_df = pd.read_sql(
    """
    SELECT id, ticker, signal_type, confidence, sentiment_score,
           politician_action, analyst_rating, created_at
    FROM signals
    ORDER BY created_at DESC
    """,
    conn,
)
signals_df["created_at"] = pd.to_datetime(signals_df["created_at"])
print(f"{len(signals_df)} total signals")
signals_df.head(20)

In [ ]:
fig = px.histogram(
    signals_df,
    x="confidence",
    color="signal_type",
    nbins=20,
    title="Confidence score distribution by signal type",
    template="plotly_white",
    barmode="overlay",
    opacity=0.7,
)
fig.add_vline(x=0.65, line_dash="dash", line_color="grey", annotation_text="min_confidence=0.65")
fig.show()

## 3. Orders and P&L

In [ ]:
orders_df = pd.read_sql(
    """
    SELECT o.id, o.ticker, o.order_type, o.qty, o.entry_price, o.stop_price,
           o.status, o.pnl, o.submitted_at, o.closed_at,
           s.confidence, s.sentiment_score, s.politician_action, s.analyst_rating
    FROM orders o
    LEFT JOIN signals s ON o.signal_id = s.id
    WHERE o.status != 'ghost'
    ORDER BY o.submitted_at DESC
    """,
    conn,
)
orders_df["submitted_at"] = pd.to_datetime(orders_df["submitted_at"])
orders_df["closed_at"] = pd.to_datetime(orders_df["closed_at"])
print(f"{len(orders_df)} non-ghost orders")
orders_df.head(20)

In [ ]:
closed = orders_df[orders_df["status"] == "closed"].copy()
open_pos = orders_df[orders_df["status"] == "open"].copy()

print(f"Open positions:  {len(open_pos)}")
print(f"Closed positions: {len(closed)}")

if len(closed) > 0:
    total_pnl = closed["pnl"].sum()
    win_rate = (closed["pnl"] > 0).mean()
    avg_win = closed.loc[closed["pnl"] > 0, "pnl"].mean()
    avg_loss = closed.loc[closed["pnl"] <= 0, "pnl"].mean()
    print(f"\nClosed trade summary:")
    print(f"  Total realised P&L: ${total_pnl:+,.2f}")
    print(f"  Win rate:           {win_rate:.1%}")
    print(f"  Average win:        ${avg_win:,.2f}")
    print(f"  Average loss:       ${avg_loss:,.2f}")
    print(f"  Expectancy:         ${closed['pnl'].mean():,.2f} per trade")

In [ ]:
if len(closed) > 0:
    closed_sorted = closed.sort_values("closed_at")
    closed_sorted["cumulative_pnl"] = closed_sorted["pnl"].cumsum()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=closed_sorted["closed_at"],
        y=closed_sorted["cumulative_pnl"],
        mode="lines+markers",
        name="Cumulative P&L",
        line=dict(color="royalblue", width=2),
    ))
    fig.add_hline(y=0, line_dash="dash", line_color="grey")
    fig.update_layout(
        title="Cumulative realised P&L over time",
        xaxis_title="Date",
        yaxis_title="Cumulative P&L ($)",
        template="plotly_white",
    )
    fig.show()
else:
    print("No closed trades yet.")

## 4. Signal quality: does higher confidence win more?

In [ ]:
if len(closed) > 0:
    closed["won"] = closed["pnl"] > 0
    closed["confidence_bucket"] = pd.cut(
        closed["confidence"],
        bins=[0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 1.01],
        labels=["0.60-0.65", "0.65-0.70", "0.70-0.75", "0.75-0.80",
                "0.80-0.85", "0.85-0.90", "0.90+"],
    )
    quality = closed.groupby("confidence_bucket", observed=True).agg(
        trades=("won", "count"),
        win_rate=("won", "mean"),
        avg_pnl=("pnl", "mean"),
    ).reset_index()

    fig = px.bar(
        quality,
        x="confidence_bucket",
        y="win_rate",
        text=quality["trades"].map(lambda n: f"n={n}"),
        title="Win rate by confidence bucket",
        labels={"confidence_bucket": "Confidence range", "win_rate": "Win rate"},
        template="plotly_white",
        color="win_rate",
        color_continuous_scale="RdYlGn",
    )
    fig.add_hline(y=0.5, line_dash="dash", line_color="grey", annotation_text="50% baseline")
    fig.update_yaxes(tickformat=".0%")
    fig.show()
    display(quality)
else:
    print("No closed trades yet.")

## 5. Source attribution: which signal component drives wins?

In [ ]:
if len(closed) > 0:
    # Categorise each trade by which signals were present
    def signal_source(row):
        has_sentiment = pd.notna(row["sentiment_score"]) and abs(row["sentiment_score"]) > 0.5
        has_politician = pd.notna(row["politician_action"])
        has_analyst = pd.notna(row["analyst_rating"]) and row["analyst_rating"] in ("Strong Buy", "Strong Sell")
        sources = []
        if has_sentiment:  sources.append("sentiment")
        if has_politician: sources.append("politician")
        if has_analyst:    sources.append("analyst")
        return " + ".join(sources) if sources else "none"

    closed["sources"] = closed.apply(signal_source, axis=1)
    attribution = closed.groupby("sources").agg(
        trades=("won", "count"),
        win_rate=("won", "mean"),
        total_pnl=("pnl", "sum"),
    ).reset_index().sort_values("win_rate", ascending=False)

    display(attribution)
else:
    print("No closed trades yet.")

## 6. Ghost trade regret: what did we miss by not executing?

In [ ]:
ghost_df = pd.read_sql(
    """
    SELECT p.date, p.ticker, p.mark_price, p.unrealized_pnl,
           s.signal_type, s.confidence
    FROM performance p
    LEFT JOIN orders o ON p.order_id = o.id
    LEFT JOIN signals s ON o.signal_id = s.id
    WHERE p.is_ghost = 1
    ORDER BY p.date DESC
    """,
    conn,
)
print(f"{len(ghost_df)} ghost trade performance records")

if len(ghost_df) > 0:
    total_ghost_pnl = ghost_df["unrealized_pnl"].sum()
    print(f"Total P&L missed by not executing ghost trades: ${total_ghost_pnl:+,.2f}")
    ghost_df.head(20)

## 7. Pending approvals history

In [ ]:
approvals_df = pd.read_sql(
    """
    SELECT pa.id, pa.ticker, pa.signal_type, pa.status,
           pa.created_at, pa.resolved_at,
           s.confidence
    FROM pending_approvals pa
    LEFT JOIN signals s ON pa.signal_id = s.id
    ORDER BY pa.created_at DESC
    """,
    conn,
)
print(f"{len(approvals_df)} approval records")

if len(approvals_df) > 0:
    print("\nStatus breakdown:")
    print(approvals_df["status"].value_counts().to_string())
    display(approvals_df)

## 8. Adanos API budget

In [ ]:
budget_df = pd.read_sql(
    "SELECT month, call_count FROM adanos_usage ORDER BY month DESC",
    conn,
)
budget_df["remaining"] = 225 - budget_df["call_count"]
budget_df["pct_used"] = (budget_df["call_count"] / 225 * 100).round(1)
display(budget_df)

if len(budget_df) > 0:
    fig = px.bar(
        budget_df,
        x="month",
        y=["call_count", "remaining"],
        title="Adanos API usage vs monthly budget (225 calls)",
        labels={"value": "API calls", "variable": ""},
        template="plotly_white",
        barmode="stack",
        color_discrete_map={"call_count": "steelblue", "remaining": "lightgrey"},
    )
    fig.show()

In [ ]:
conn.close()
print("Connection closed.")